In [2]:
### 文本擷取

import fitz  # PyMuPDF
import re


def step1_extract_text(pdf_path):
    """
    從 PDF 中提取文字，並進行初步的格式清理。
    """
    try:
        # 開啟 PDF 檔案
        doc = fitz.open(pdf_path)
        print(f"--- 檔案讀取成功：{pdf_path} ---")
        print(f"總頁數: {len(doc)}")
        
        extracted_data = []

        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            # 提取文字
            raw_text = page.get_text("text")
            
            # 初步清理：移除多餘的連續空白、統一換行符
            clean_text = re.sub(r'\n\s*\n', '\n\n', raw_text) # 保持段落感
            clean_text = clean_text.strip()
            
            # 儲存結果（包含頁碼資訊，這對後續 RAG 引用非常重要）
            extracted_data.append({
                "page": page_num + 1,
                "content": clean_text
            })
            
            # 預覽前兩頁
            if page_num < 2:
                print(f"\n[第 {page_num + 1} 頁預覽]:")
                print(clean_text[:300] + "...") 
                print("-" * 30)

        doc.close()
        return extracted_data

    except Exception as e:
        print(f"讀取失敗：{e}")
        return None

# --- 執行處 ---
# 請將 'HIWIN_Catalog.pdf' 換成你實際的檔案路徑

raw_pages = step1_extract_text(R"C:\Users\e11338\Desktop\銀泰目錄分割\HIWIN 精密研磨級滾珠螺桿系列 切割 45_49(頁數).pdf")

--- 檔案讀取成功：C:\Users\e11338\Desktop\銀泰目錄分割\HIWIN 精密研磨級滾珠螺桿系列 切割 45_49(頁數).pdf ---
總頁數: 5

[第 1 頁預覽]:
S99TC13-1304 41
Type
規格品
6.2 精密研磨級滾珠螺桿尺寸
F
S
V
型號
規格
珠徑
PCD
根徑
珠卷數
剛性
Kgf /μm
K
動負荷
C ( kgf )
靜負荷
Co ( kgf )
螺帽
法蘭
 迴流管
法蘭孔
接觸
面長
公稱
外徑
導程
D
L
F
T
BCD-E
W
H
X
Y
Z
S
16-4B2
16       
4
2.381
16.25
13.792
2.5x2
26
802
1722
30
48
52
10
40
23
21
5.5
9.5
5.5
12
16-5B1
5
3.175
16.6
13.324
2.5x1
16
763
140...
------------------------------

[第 2 頁預覽]:
S99TC13-1304
42
Type
規格品
F
S
V
ØF 
ØD 
-0.1 
-0.3 
ØDg6 
30° 
30° 
Wmax 
Hmax 
BCD E 
ØX 
ØY 
L 
Z 
S 
T 
油孔
T<12  M6x1P
T≥12  1/8PT
型號
規格
珠徑
PCD
根徑
珠卷數
剛性
Kgf /μm
K
動負荷
C ( kgf )
靜負荷
Co ( kgf )
螺帽
法蘭
 迴流管
法蘭孔
接觸
面長
公稱
外徑導程
D
L
F
T
BCD-E
W
H
X
Y
Z
S
36-10B2
36
10
6.350
37.4
30.91
2.5x2
68
5105
12669...
------------------------------


In [ ]:
### 文本擷取，文字重新排列

import fitz
import pdfplumber
import pandas as pd
import re

#pdf_path = R"C:\Users\e11338\Desktop\銀泰目錄分割\HIWIN 精密研磨級滾珠螺桿系列 切割 45_49(頁數).pdf"
pdf_path = R"C:\Users\e11338\Desktop\銀泰目錄分割\HIWIN 精密研磨級滾珠螺桿系列 切割 45_146(頁數).pdf"
doc_fitz = fitz.open(pdf_path)

with pdfplumber.open(pdf_path) as pdf_plumb:
        for i in range(len(pdf_plumb.pages)):
            page_fitz = doc_fitz[i]
            page = doc_fitz.load_page(i)
            
            # --- 關鍵修正：強制按「空間座標」排序文字 ---
            # get_text("words") 會回傳 (x0, y0, x1, y1, "word", ...)
            words = page_fitz.get_text("words")
            # 排序邏輯：優先比 y0 (高度)，y0 相近時(誤差3像素內)比 x0 (左右)
            words.sort(key=lambda w: (w[1] // 3, w[0]))
            
            # 重新組合排序後的文字列表
            sorted_text_list = [w[4] for w in words]
            full_sorted_text = "".join(sorted_text_list) # 拼成一個長字串

            if i < 20:
                print(f"\n[第 {i+1} 頁預覽]:")
                print(full_sorted_text[:300] + "...") 
                print("-" * 30)


[第 1 頁預覽]:
41S99TC13-13046.2精密研磨級滾珠螺桿尺寸FSVType規格品LTSHmaxZØXT<12M6x1PT≥121/8PT油孔ØYBCDEWmaxØDg6ØD-0.1-0.330°30°ØF接觸規格螺帽法蘭迴流管法蘭孔剛性動負荷靜負荷面長型號珠徑PCD根徑珠卷數Kgf/μmC(kgf)Co(kgf)公稱導程KDLFTBCD-EWHXYZS外徑16-4B242.38116.2513.7922.5x2268021722304852104023215.59.55.51216-5B116.613.3242.5x1167631400314554124127225.59.55.51216-5B2...
------------------------------

[第 2 頁預覽]:
42S99TC13-1304FSVType規格品LTSHmaxZØXT<12M6x1PT≥121/8PT油孔ØYBCDEWmaxØDg6ØD-0.1-0.330°30°ØF接觸規格螺帽法蘭迴流管法蘭孔剛性動負荷靜負荷面長型號珠徑PCD根徑珠卷數Kgf/μmC(kgf)Co(kgf)公稱外徑導程KDLFTBCD-EWHXYZS36-10B236106.35037.430.912.5x26851051266962102104188249401117.5111540-5B253.17540.637.3242.5x26620717134586592167246349148.51540-6B263.96...
------------------------------

[第 3 頁預覽]:
43S99TC13-1304FSVType規格品LTSHmaxZØXT<12M6x1PT≥121/8PT油孔ØYBCDEWmaxØDg6ØD-0.1-0.330°30°ØF接觸規格螺帽法蘭迴流管法蘭孔剛性動負荷靜負荷面長型號珠徑PCD根徑珠卷數Kgf/μmC(kgf)Co(kgf)公稱外徑導程KDLFTBCD-EWHXYZS63-20B3632012.7006653.162.5x321030715908871172441573213782701117.5113070-10B271.464.912.5x21156843250111041091522012880561320132010

In [ ]:
#初步表格清洗 以及 系列名擷取

import fitz  # PyMuPDF
import pdfplumber
import pandas as pd
import re
from IPython.display import display, HTML

# 檔案路徑
pdf_path = r"C:\Users\e11338\Desktop\銀泰目錄分割\HIWIN 精密研磨級滾珠螺桿系列 切割 1_33 99_102(頁數).pdf"

# 型號匹配規則
PREFIXES = ['F', 'R', 'PF', 'OF', 'DF']
SERIES_CODES = ['SV', 'SW', 'DV', 'DW', 'SI', 'DI', 'SH', 'SC', 'DC']
SERIES_PATTERN = f"({'|'.join(PREFIXES)})({'|'.join(SERIES_CODES)})"

def clean_and_merge_table(df, series_name, page_num):
    if df is None or df.empty:
        return None
    
    # 移除換行
    df = df.replace('\n', '', regex=True)
    
    if len(df) > 2:
        new_columns = []
        for i in range(len(df.columns)):
            h1 = str(df.iloc[0, i]) if df.iloc[0, i] and df.iloc[0, i] != 'None' else ""
            h2 = str(df.iloc[1, i]) if df.iloc[1, i] and df.iloc[1, i] != 'None' else ""
            combined = f"{h1}{h2}".strip()
            if not combined:
                combined = f"Unnamed_{i}"
            new_columns.append(combined)
        
        # 處理重複欄位名 (解決 InvalidIndexError)
        final_cols = []
        counts = {}
        for col in new_columns:
            if col in counts:
                counts[col] += 1
                final_cols.append(f"{col}_{counts[col]}")
            else:
                counts[col] = 0
                final_cols.append(col)
        
        df.columns = final_cols
        df = df.iloc[2:].reset_index(drop=True)
    
    # 插入識別欄位
    df.insert(0, "Series_Name", series_name)
    df.insert(1, "Source_Page", page_num)
    
    # 排除標題殘影行
    df = df[~df.iloc[:, 2].str.contains(r'^[A-Za-z]$', na=False)]
    return df

def process_and_view():
    all_processed_tables = []
    last_known_series = "Unknown"
    
    doc_fitz = fitz.open(pdf_path)
    with pdfplumber.open(pdf_path) as pdf_plumb:
        pages_to_process = range(len(pdf_plumb.pages)) 
        
        for i in pages_to_process:
            # --- 系列名擷取 (空間排序邏輯) ---
            page_fitz = doc_fitz[i]
            words = page_fitz.get_text("words")
            words.sort(key=lambda w: (w[1] // 3, w[0]))
            full_sorted_text = "".join([w[4] for w in words])
            
            series_name = "Unknown"
            search_match = re.search(f"({SERIES_PATTERN})Type", full_sorted_text, re.IGNORECASE)
            if search_match:
                series_name = search_match.group(1)
            else:
                search_match = re.search(SERIES_PATTERN, full_sorted_text)
                if search_match:
                    series_name = search_match.group(0)
            
            if series_name == "Unknown":
                series_name = last_known_series
            else:
                last_known_series = series_name
            
            # --- 表格擷取 ---
            page_plumb = pdf_plumb.pages[i]
            table = page_plumb.extract_table({
                "vertical_strategy": "lines",
                "horizontal_strategy": "lines",
                "snap_tolerance": 3,
                "join_tolerance": 2,
            })
            
            if table:
                raw_df = pd.DataFrame(table)
                clean_df = clean_and_merge_table(raw_df, series_name, i)
                
                if clean_df is not None:
                    all_processed_tables.append(clean_df)
                    # if i < 20:
                    #     #在 Jupyter 裡面即時顯示每一頁的結果
                    #     print(f"--- 第 {i+45} 頁 (系列: {series_name}) ---")
                    #     display(clean_df.head(3)) # 只顯示前三行縮小版面
            
    # 最後合併顯示總表
    if all_processed_tables:
        return all_processed_tables 
    else:
        print("未擷取到表格。")
        return None

# 執行並取得最終 DataFrame
final_hiwin_df = process_and_view()

In [ ]:
#表格數值轉換(字串-->數字)、columns_name轉換、文意欄位新增、存檔
import pandas as pd

# 1. 定義欄位名稱
col_name = ["系列", "sorce page", "型號", "公稱 外徑", "導程", "珠徑", "PCD", "根徑", "珠卷數", "剛性 kfg/umk","動負荷 C (kfg)" , "靜負荷 Co (kfg)"]
numeric_cols = ["公稱 外徑", "導程", "珠徑", "PCD", "根徑", "剛性 kfg/umk","動負荷 C (kfg)" , "靜負荷 Co (kfg)"]

# 2. 合併所有的 DataFrame
processed_list = []
for df_single in final_hiwin_df:
    temp_df = df_single.iloc[:, :12].copy()
    temp_df.columns = col_name
    processed_list.append(temp_df)

final_hiwin = pd.concat(processed_list, ignore_index=True)

# --- 關鍵清洗步驟：移除標題列 ---
# 只要「公稱 外徑」這一欄的內容剛好等於字串 "公稱 外徑"，就代表它是標題殘影
final_hiwin = final_hiwin[final_hiwin["公稱 外徑"] != "公稱 外徑"].reset_index(drop=True)

# 3. 轉換資料型態 (to_numeric)
def change_astype_robust(df, cols):
    for c in cols:
        # errors='coerce' 會把無法轉成數字的文字（如標題或空白）變成 NaN
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

final_hiwin = change_astype_robust(final_hiwin, numeric_cols)

# 4. 向下填充 (處理合併儲存格產生的空白，需在轉數字後執行)
final_hiwin = final_hiwin.ffill()

# 5. 生成語義文本 (加入檢查以確保數值正確)
def generate_semantic(row):
    return (
        f"這是上銀 (HIWIN) 的滾珠螺桿規格。系列名稱為 {row['系列']}，"
        f"完整型號為 {row['型號']}。其主要參數如下：公稱外徑為 {row['公稱 外徑']} mm，"
        f"導程為 {row['導程']} mm，珠徑為 {row['珠徑']} mm，珠卷數為 {row['珠卷數']}。"
        f"在性能指標方面，其動負荷 (Ca) 為 {row['動負荷 C (kfg)']} kgf，"
        f"靜負荷 (Co) 為 {row['靜負荷 Co (kfg)']} kgf，剛性為 {row['剛性 kfg/umk']} kgf/umk。"
    )

final_hiwin['semantic_text'] = final_hiwin.apply(generate_semantic, axis=1)

output_file = "HIWIN_Final_Data_V1.xlsx"
final_hiwin.to_excel(output_file, index=False)

# 預覽結果
print("資料清洗完成！")
display(final_hiwin.head())

資料清洗完成！


,系列,sorce page,型號,公稱 外徑,導程,珠徑,PCD,根徑,珠卷數,剛性 kfg/umk,動負荷 C (kfg),靜負荷 Co (kfg),semantic_text
0,FSV,0,16-4B2,16.0,4.0,2.381,16.25,13.792,2.5x2,26.0,802.0,1722.0,這是上銀 (HIWIN) 的滾珠螺桿規格。系列名稱為 FSV，完整型號為 16-4B2。其主...
1,FSV,0,16-5B1,16.0,5.0,3.175,16.60,13.324,2.5x1,16.0,763.0,1400.0,這是上銀 (HIWIN) 的滾珠螺桿規格。系列名稱為 FSV，完整型號為 16-5B1。其主...
2,FSV,0,16-5B2,16.0,5.0,3.175,16.60,13.324,2.5x2,33.0,1385.0,2799.0,這是上銀 (HIWIN) 的滾珠螺桿規格。系列名稱為 FSV，完整型號為 16-5B2。其主...
3,FSV,0,16-5C1,16.0,5.0,3.175,16.60,13.324,3.5x1,22.0,1013.0,1946.0,這是上銀 (HIWIN) 的滾珠螺桿規格。系列名稱為 FSV，完整型號為 16-5C1。其主...
4,FSV,0,16-10B1,16.0,10.0,3.175,16.60,13.324,2.5x1,16.0,763.0,1399.0,這是上銀 (HIWIN) 的滾珠螺桿規格。系列名稱為 FSV，完整型號為 16-10B1。其...
